# Chapter 18: Capacity Checks and Equipment Utilization

This notebook demonstrates how to use NeqSim's capacity analysis framework to:
- Build a process with separator, compressor, and pipeline
- Use `autoSize()` to set design constraints from operating conditions
- Query capacity utilization summaries from `ProcessSystem`
- Detect bottlenecks with `findBottleneck()`
- Visualize utilization across equipment and sweep feed rates

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 18.1 Build a Simple Gas Processing Flowsheet

We create a feed stream of natural gas, pass it through a separator, compress the gas outlet,
and send it through an export pipeline.

In [2]:
from neqsim import jneqsim

# Create fluid
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 25.0, 50.0)
fluid.addComponent("nitrogen", 0.02)
fluid.addComponent("CO2", 0.03)
fluid.addComponent("methane", 0.80)
fluid.addComponent("ethane", 0.06)
fluid.addComponent("propane", 0.04)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("n-pentane", 0.01)
fluid.addComponent("n-hexane", 0.01)
fluid.addComponent("water", 0.01)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

# Build process
feed = jneqsim.process.equipment.stream.Stream("Feed Gas", fluid)
feed.setFlowRate(50000.0, "kg/hr")
feed.setTemperature(25.0, "C")
feed.setPressure(50.0, "bara")

separator = jneqsim.process.equipment.separator.Separator("HP Separator", feed)

compressor = jneqsim.process.equipment.compressor.Compressor("Export Compressor", separator.getGasOutStream())
compressor.setOutletPressure(120.0)

cooler = jneqsim.process.equipment.heatexchanger.Cooler("After-Cooler", compressor.getOutletStream())
cooler.setOutTemperature(273.15 + 35.0)

pipeline = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Export Pipeline", cooler.getOutletStream())
pipeline.setPipeWallRoughness(5e-5)
pipeline.setLength(50.0)
pipeline.setElevation(0.0)
pipeline.setDiameter(0.3)

process = jneqsim.process.processmodel.ProcessSystem()
process.add(feed)
process.add(separator)
process.add(compressor)
process.add(cooler)
process.add(pipeline)
process.run()

print(f"Feed rate: {feed.getFlowRate('kg/hr'):.0f} kg/hr")
print(f"Compressor power: {compressor.getPower('kW'):.1f} kW")
print(f"Pipeline outlet pressure: {pipeline.getOutletStream().getPressure('bara'):.2f} bara")

Feed rate: 50000 kg/hr
Compressor power: 1345.9 kW
Pipeline outlet pressure: 120.00 bara


## 18.2 Auto-Sizing Equipment

The `autoSize()` method sets design capacity constraints from current operating conditions
with a specified safety factor. This establishes the equipment's rated capacity for
subsequent utilization checks.

In [3]:
# Auto-size equipment with a 1.2 safety factor (20% margin above current operation)
separator.autoSize(1.2)
compressor.autoSize(1.2)
pipeline.autoSize(1.2)

print(f"Separator auto-sized: {separator.isAutoSized()}")
print(f"Compressor auto-sized: {compressor.isAutoSized()}")
print(f"Pipeline auto-sized: {pipeline.isAutoSized()}")

# Re-run to propagate
process.run()
print("\nProcess re-run after auto-sizing complete.")

Separator auto-sized: True
Compressor auto-sized: True
Pipeline auto-sized: True



Process re-run after auto-sizing complete.


## 18.3 Capacity Utilization Summary

After auto-sizing, we can query the `ProcessSystem` for a utilization summary across
all constrained equipment. Values are percentages — 83% means the equipment is using
83% of its design capacity (since we sized with a 1.2 factor, nominal is ~83%).

In [4]:
# Get utilization summary
utilization_map = process.getCapacityUtilizationSummary()

print("=" * 50)
print(f"{'Equipment':<30} {'Utilization (%)':<15}")
print("=" * 50)
for entry in utilization_map.entrySet():
    name = str(entry.getKey())
    util = float(entry.getValue())
    print(f"{name:<30} {util:<15.1f}")
print("=" * 50)

Equipment                      Utilization (%)


HP Separator                   83.1           
Export Compressor              108.9          
Export Pipeline                28.9           


## 18.4 Bottleneck Detection

The `findBottleneck()` method returns a `BottleneckResult` identifying which equipment
and which specific constraint is most limiting.

In [5]:
bottleneck = process.findBottleneck()

if bottleneck.hasBottleneck():
    print(f"Bottleneck equipment: {bottleneck.getEquipmentName()}")
    print(f"Limiting constraint:  {bottleneck.getConstraintName()}")
    print(f"Utilization:          {bottleneck.getUtilizationPercent():.1f}%")
else:
    print("No bottleneck detected (no constrained equipment).")

# Check if anything is overloaded
print(f"\nAny equipment overloaded? {process.isAnyEquipmentOverloaded()}")

Bottleneck equipment: Export Compressor
Limiting constraint:  power
Utilization:          108.9%

Any equipment overloaded? True


## 18.5 Utilization Bar Chart

Visualize the utilization of all constrained equipment as a horizontal bar chart.
The 100% line marks the design capacity boundary.

In [6]:
# Extract data from Java map
names = []
utils = []
for entry in utilization_map.entrySet():
    names.append(str(entry.getKey()))
    utils.append(float(entry.getValue()))

if len(names) > 0:
    colors = ['green' if u < 80 else 'orange' if u < 100 else 'red' for u in utils]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(names, utils, color=colors, edgecolor='black', linewidth=0.5)
    ax.axvline(x=100, color='red', linestyle='--', linewidth=1.5, label='Design Capacity (100%)')
    ax.set_xlabel('Capacity Utilization (%)', fontsize=12)
    ax.set_title('Equipment Capacity Utilization at Nominal Feed Rate', fontsize=13)
    ax.legend(loc='lower right')
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(0, max(max(utils) * 1.15, 110))

    # Add value labels
    for bar, val in zip(bars, utils):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=10)

    plt.tight_layout()
    plt.savefig("../figures/ch18_utilization_bar_chart.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No constrained equipment found. Auto-size may not have applied constraints.")

C:\Users\ESOL\AppData\Local\Temp\ipykernel_40860\1003577224.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 18.6 Feed Rate Sweep — Utilization Curves

We sweep the feed flow rate from 20,000 to 80,000 kg/hr and plot how each equipment's
utilization changes. This reveals which equipment becomes the bottleneck first as
production increases.

In [7]:
flow_rates = np.linspace(20000, 80000, 15)
results = {}
bottleneck_names = []

for rate in flow_rates:
    feed.setFlowRate(float(rate), "kg/hr")
    try:
        process.run()
    except Exception:
        break

    util_map = process.getCapacityUtilizationSummary()
    for entry in util_map.entrySet():
        name = str(entry.getKey())
        val = float(entry.getValue())
        if name not in results:
            results[name] = []
        results[name].append(val)

    bn = process.findBottleneck()
    bottleneck_names.append(bn.getEquipmentName() if bn.hasBottleneck() else "None")

# Reset to nominal
feed.setFlowRate(50000.0, "kg/hr")
process.run()

# Plot
n_points = min(len(flow_rates), min(len(v) for v in results.values()) if results else 0)

if n_points > 0 and len(results) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, vals in results.items():
        ax.plot(flow_rates[:n_points] / 1000, vals[:n_points], 'o-', label=name, linewidth=2)

    ax.axhline(y=100, color='red', linestyle='--', linewidth=1.5, label='Design Capacity')
    ax.set_xlabel('Feed Rate (1000 kg/hr)', fontsize=12)
    ax.set_ylabel('Capacity Utilization (%)', fontsize=12)
    ax.set_title('Equipment Utilization vs Feed Rate', fontsize=13)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(150, max(max(v[:n_points]) for v in results.values()) * 1.1))

    plt.tight_layout()
    plt.savefig("../figures/ch18_utilization_vs_feed_rate.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nBottleneck at each flow rate:")
    for i in range(n_points):
        print(f"  {flow_rates[i]/1000:.0f} t/hr -> {bottleneck_names[i]}")
else:
    print("Insufficient data for utilization sweep plot.")


Bottleneck at each flow rate:
  20 t/hr -> Export Compressor
  24 t/hr -> Export Compressor
  29 t/hr -> Export Compressor
  33 t/hr -> Export Compressor
  37 t/hr -> Export Compressor
  41 t/hr -> Export Compressor
  46 t/hr -> Export Compressor
  50 t/hr -> Export Compressor
  54 t/hr -> Export Compressor
  59 t/hr -> Export Compressor
  63 t/hr -> Export Compressor
  67 t/hr -> Export Compressor
  71 t/hr -> Export Compressor
  76 t/hr -> Export Compressor
  80 t/hr -> Export Compressor


C:\Users\ESOL\AppData\Local\Temp\ipykernel_40860\295831666.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

Key takeaways from this chapter:

1. **Auto-sizing** (`autoSize(factor)`) establishes design constraints from current operating conditions
2. **`getCapacityUtilizationSummary()`** provides a quick overview of all equipment utilization
3. **`findBottleneck()`** identifies the specific equipment and constraint limiting production
4. **Feed rate sweeps** reveal how bottlenecks shift as production changes
5. Equipment utilization near 80-90% of design is typical for normal operation; above 100% indicates overload